# Étape 1 — Nettoyage & Feature Engineering
## Projet MLOps — Home Credit Default Risk

**Fichiers utilisés :**
- `application_train.csv` → données d'entraînement (avec TARGET)
- `application_test.csv` → données de test (sans TARGET)

## 0. Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings('ignore')

DATA_PATH   = '/home/veron/Documents/OpenClassRoom/p6/dataset/'
OUTPUT_PATH = '/home/veron/Documents/OpenClassRoom/p6/dataset/'

print('✅ Imports OK')

✅ Imports OK


## 1. Chargement & Exploration

In [2]:
train = pd.read_csv(DATA_PATH + 'application_train.csv')
test  = pd.read_csv(DATA_PATH + 'application_test.csv')

print(f'Train : {train.shape[0]:,} lignes × {train.shape[1]} colonnes')
print(f'Test  : {test.shape[0]:,} lignes  × {test.shape[1]} colonnes')
print(f'\nDistribution TARGET :')
print(train['TARGET'].value_counts(normalize=True).round(3))
print(f'\nDéséquilibre : {train["TARGET"].mean()*100:.1f}% de défauts')

Train : 307,511 lignes × 122 colonnes
Test  : 48,744 lignes  × 121 colonnes

Distribution TARGET :
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64

Déséquilibre : 8.1% de défauts


## 2. Fusion Train + Test pour un preprocessing cohérent

In [3]:
# Fusion pour appliquer le même preprocessing aux deux
df = pd.concat([train, test], ignore_index=True)
del train, test; gc.collect()

print(f'Dataset fusionné : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

Dataset fusionné : 356,255 lignes × 122 colonnes


## 3. Nettoyage

In [4]:
# Supprimer colonnes avec >40% de NaN
pct_missing = df.isnull().sum() / len(df)
cols_to_drop = pct_missing[pct_missing > 0.4].index.tolist()
df = df.drop(columns=cols_to_drop)
print(f'Colonnes supprimées (>40% NaN) : {len(cols_to_drop)}')

# Correction anomalie DAYS_EMPLOYED
df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

# Supprimer CODE_GENDER = XNA
df = df[df['CODE_GENDER'] != 'XNA']

# Encodage binaire
for col in ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']:
    df[col], _ = pd.factorize(df[col])

# One-hot encoding des variables catégorielles
cat_cols = [c for c in df.columns if df[c].dtype == 'object']
df = pd.get_dummies(df, columns=cat_cols)

# Remplacer les infinis par NaN
df = df.replace([np.inf, -np.inf], np.nan)

print(f'Shape après nettoyage : {df.shape}')

Colonnes supprimées (>40% NaN) : 49
Shape après nettoyage : (356251, 181)


## 4. Features métier

In [5]:
df['INCOME_CREDIT_PERC']  = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']
df['INCOME_PER_PERSON']   = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
df['ANNUITY_INCOME_PERC'] = df['AMT_ANNUITY']      / df['AMT_INCOME_TOTAL']
df['PAYMENT_RATE']        = df['AMT_ANNUITY']      / df['AMT_CREDIT']
df['DAYS_EMPLOYED_PERC']  = df['DAYS_EMPLOYED']    / df['DAYS_BIRTH']

# Remplacer les infinis générés par les divisions
df = df.replace([np.inf, -np.inf], np.nan)

print(f'Features finales : {df.shape[1]} colonnes')

Features finales : 186 colonnes


## 5. Séparation Train / Test & Sauvegarde

In [6]:
train_df = df[df['TARGET'].notnull()].copy()
test_df  = df[df['TARGET'].isnull()].copy()
del df; gc.collect()

print(f'Train : {train_df.shape[0]:,} lignes × {train_df.shape[1]} colonnes')
print(f'Test  : {test_df.shape[0]:,} lignes  × {test_df.shape[1]} colonnes')

# Sauvegarde en parquet (léger et rapide)
print('\nSauvegarde en cours...')
train_df.to_parquet(OUTPUT_PATH + 'df_train_enrichi.parquet', index=False)
print(f'✅ Train sauvegardé → df_train_enrichi.parquet')

test_df.to_parquet(OUTPUT_PATH + 'df_test_enrichi.parquet', index=False)
print(f'✅ Test  sauvegardé → df_test_enrichi.parquet')
print(f'\n📁 {OUTPUT_PATH}')

Train : 307,507 lignes × 186 colonnes
Test  : 48,744 lignes  × 186 colonnes

Sauvegarde en cours...
✅ Train sauvegardé → df_train_enrichi.parquet
✅ Test  sauvegardé → df_test_enrichi.parquet

📁 /home/veron/Documents/OpenClassRoom/p6/dataset/


---
## ✅ Checklist — Étape 1

| Indicateur | Statut |
|---|---|
| Exploration données brutes | ✅ |
| Fusion train + test cohérente | ✅ |
| Suppression colonnes >40% NaN | ✅ |
| Correction anomalie DAYS_EMPLOYED | ✅ |
| Encodage variables catégorielles | ✅ |
| Nouvelles features métier | ✅ |
| Remplacement valeurs infinies | ✅ |
| Datasets sauvegardés en parquet | ✅ |